In [3]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

from src.data.syntethic.generate import generate
from src.data.preprocess import preprocess
from src.config.config import FEATURES, TARGET

raw_df = generate(n=3000, seed=42, save=False)
df     = preprocess(raw_df, save=False)

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train)} | Test size: {len(X_test)}')
print(f'No-show rate in test: {y_test.mean():.2%}')

Train size: 2400 | Test size: 600
No-show rate in test: 23.33%


In [4]:
def evaluate(name, model, X_test, y_test, threshold=0.5):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cm  = confusion_matrix(y_test, y_pred)

    print(f'\n{"-"*50}')
    print(f'  EXPERIMENT: {name}')
    print(f'  Threshold used: {threshold}')
    print(f'{"-"*50}')
    print(f'  Accuracy:      {acc:.4f}')
    print(f'  ROC-AUC:       {auc:.4f}')
    print(f'  F1 (No-Show):  {f1:.4f}')
    print(f'  Confusion Matrix:')
    print(f'    TN={cm[0][0]}  FP={cm[0][1]}')
    print(f'    FN={cm[1][0]}  TP={cm[1][1]}')
    print(f'\n{classification_report(y_test, y_pred, target_names=["Show", "No-Show"])}')

    return {'name': name, 'accuracy': acc, 'f1': f1, 'roc_auc': auc, 'threshold': threshold}

results = []
print('Helper ready!')

Helper ready!


## Experiment 1 Baseline (class_weight=balanced, threshold=0.5)

In [5]:
model_v1 = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model_v1.fit(X_train, y_train)

r1 = evaluate('v1 - balanced, threshold=0.5', model_v1, X_test, y_test, threshold=0.5)
results.append(r1)


--------------------------------------------------
  EXPERIMENT: v1 - balanced, threshold=0.5
  Threshold used: 0.5
--------------------------------------------------
  Accuracy:      0.7567
  ROC-AUC:       0.5494
  F1 (No-Show):  0.1205
  Confusion Matrix:
    TN=444  FP=16
    FN=130  TP=10

              precision    recall  f1-score   support

        Show       0.77      0.97      0.86       460
     No-Show       0.38      0.07      0.12       140

    accuracy                           0.76       600
   macro avg       0.58      0.52      0.49       600
weighted avg       0.68      0.76      0.69       600



# Experiment 2  Lower Threshold to 0.30
Same model as v1, but we lower the decision bar for predicting no-show.

In [6]:
r2 = evaluate('v2 - balanced, threshold=0.30', model_v1, X_test, y_test, threshold=0.30)
results.append(r2)


--------------------------------------------------
  EXPERIMENT: v2 - balanced, threshold=0.30
  Threshold used: 0.3
--------------------------------------------------
  Accuracy:      0.6250
  ROC-AUC:       0.5494
  F1 (No-Show):  0.2902
  Confusion Matrix:
    TN=329  FP=131
    FN=94  TP=46

              precision    recall  f1-score   support

        Show       0.78      0.72      0.75       460
     No-Show       0.26      0.33      0.29       140

    accuracy                           0.62       600
   macro avg       0.52      0.52      0.52       600
weighted avg       0.66      0.62      0.64       600



## Experiment 3 Stronger Class Weight {0:1, 1:4}
We punish the model = missing a no-show is 4x worse than missing a show.

In [7]:
model_v3 = RandomForestClassifier(
    n_estimators=300,
    class_weight={0: 1, 1: 4},
    random_state=42,
    n_jobs=-1
)
model_v3.fit(X_train, y_train)

r3 = evaluate('v3 - weight {0:1,1:4}, threshold=0.30', model_v3, X_test, y_test, threshold=0.30)
results.append(r3)


--------------------------------------------------
  EXPERIMENT: v3 - weight {0:1,1:4}, threshold=0.30
  Threshold used: 0.3
--------------------------------------------------
  Accuracy:      0.6283
  ROC-AUC:       0.5447
  F1 (No-Show):  0.2591
  Confusion Matrix:
    TN=338  FP=122
    FN=101  TP=39

              precision    recall  f1-score   support

        Show       0.77      0.73      0.75       460
     No-Show       0.24      0.28      0.26       140

    accuracy                           0.63       600
   macro avg       0.51      0.51      0.51       600
weighted avg       0.65      0.63      0.64       600



## Experiment 4 using gridsearch tuned (scoring=f1)
Automatically finds the best hyperparameters


In [8]:
param_grid = {
    'n_estimators':     [200, 300, 500],
    'max_depth':        [None, 10, 20],
    'min_samples_leaf': [1, 2, 4],
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(
        class_weight={0: 1, 1: 4},
        random_state=42,
        n_jobs=-1,
    ),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)

print(f'Best params: {grid_search.best_params_}')
model_v4 = grid_search.best_estimator_

r4 = evaluate('v4 - GridSearchCV, threshold=0.30', model_v4, X_test, y_test, threshold=0.30)
results.append(r4)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best params: {'max_depth': 10, 'min_samples_leaf': 4, 'n_estimators': 200}

--------------------------------------------------
  EXPERIMENT: v4 - GridSearchCV, threshold=0.30
  Threshold used: 0.3
--------------------------------------------------
  Accuracy:      0.3633
  ROC-AUC:       0.5857
  F1 (No-Show):  0.3994
  Confusion Matrix:
    TN=91  FP=369
    FN=13  TP=127

              precision    recall  f1-score   support

        Show       0.88      0.20      0.32       460
     No-Show       0.26      0.91      0.40       140

    accuracy                           0.36       600
   macro avg       0.57      0.55      0.36       600
weighted avg       0.73      0.36      0.34       600



In [9]:
model_v5 = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    min_samples_leaf=4,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model_v5.fit(X_train, y_train)

r5 = evaluate('v5 - with best params', model_v5, X_test, y_test)
results.append(r5)


--------------------------------------------------
  EXPERIMENT: v5 - with best params
  Threshold used: 0.5
--------------------------------------------------
  Accuracy:      0.7083
  ROC-AUC:       0.5820
  F1 (No-Show):  0.3028
  Confusion Matrix:
    TN=387  FP=73
    FN=102  TP=38

              precision    recall  f1-score   support

        Show       0.79      0.84      0.82       460
     No-Show       0.34      0.27      0.30       140

    accuracy                           0.71       600
   macro avg       0.57      0.56      0.56       600
weighted avg       0.69      0.71      0.70       600



# Best params
### {'max_depth': 10, 'min_samples_leaf': 4, 'n_estimators': 200} 

In [10]:
summary = pd.DataFrame(results)
summary = summary.set_index('name')
summary = summary.round(4)
print('\n=== EXPERIMENT SUMMARY ===')
print(summary.to_string())

best = summary['f1'].idxmax()
print(f'\nBest model by F1: {best}')


=== EXPERIMENT SUMMARY ===
                                       accuracy      f1  roc_auc  threshold
name                                                                       
v1 - balanced, threshold=0.5             0.7567  0.1205   0.5494        0.5
v2 - balanced, threshold=0.30            0.6250  0.2902   0.5494        0.3
v3 - weight {0:1,1:4}, threshold=0.30    0.6283  0.2591   0.5447        0.3
v4 - GridSearchCV, threshold=0.30        0.3633  0.3994   0.5857        0.3
v5 - with best params                    0.7083  0.3028   0.5820        0.5

Best model by F1: v4 - GridSearchCV, threshold=0.30
